<a href="https://colab.research.google.com/github/nishithy13/ds2002-fa26/blob/main/notebooks/01-foundations/2026-09-30%20%E2%80%94%20JSON_to_Tidy_Tables%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## One response, straight off the wire

Monday you took apart a payload that had already been laid out for you, indented and readable. Real responses do not arrive that way. Whitespace costs bandwidth, so APIs ship their JSON with every space squeezed out of it.

Here is a portfolio response from a market-data service, exactly as it came down. Run the cell and try to read it.

In [1]:
RAW = '{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple Inc.","sector":"Technology"},"shares":120,"prices":[{"date":"2026-09-28","close":231.4,"volume":52300000},{"date":"2026-09-29","close":229.85,"volume":48100000},{"date":"2026-09-30","close":233.1,"volume":55600000}]},{"ticker":"MSFT","company":{"name":"Microsoft Corp.","sector":"Technology"},"shares":45,"prices":[{"date":"2026-09-28","close":502.1,"volume":22100000},{"date":"2026-09-29","close":"498.75","volume":0},{"date":"2026-09-30","close":505.3,"volume":24800000}]},{"ticker":"KO","company":{"name":"Coca-Cola Co."},"shares":300,"prices":[{"date":"2026-09-28","close":71.25,"volume":12400000},{"date":"2026-09-29","close":null,"volume":13000000},{"date":"2026-09-30","close":70.9,"volume":11800000}]}]}}'
print('characters:', len(RAW))
print(RAW[:160], '...')

characters: 871
{"as_of":"2026-09-30","currency":"USD","source":"teaching-fixture","portfolio":{"name":"Cavalier Growth","holdings":[{"ticker":"AAPL","company":{"name":"Apple I ...


Eight hundred and seventy-one characters on one line. You cannot read that, and neither can anyone else. That is fine. **Nobody reads raw JSON.** You parse it, then you ask Python to lay it out for you.

Notice what you are holding right now: `RAW` is a *string*. It has a length. You could slice it. There are no keys to look up yet, because text does not have keys.

### Step 1 — parse it, then make it readable

Two different jobs, and people confuse them constantly. `json.loads` turns text into Python objects. `json.dumps` with `indent` turns Python objects back into text that a human can scan. You are about to do both, in that order, and end up looking at the same data you started with — just formatted.

In [2]:
import json, pandas as pd

data = json.loads(RAW)
print(type(RAW), '->', type(data))
print(json.dumps(data, indent=2)[:700], '...')

<class 'str'> -> <class 'dict'>
{
  "as_of": "2026-09-30",
  "currency": "USD",
  "source": "teaching-fixture",
  "portfolio": {
    "name": "Cavalier Growth",
    "holdings": [
      {
        "ticker": "AAPL",
        "company": {
          "name": "Apple Inc.",
          "sector": "Technology"
        },
        "shares": 120,
        "prices": [
          {
            "date": "2026-09-28",
            "close": 231.4,
            "volume": 52300000
          },
          {
            "date": "2026-09-29",
            "close": 229.85,
            "volume": 48100000
          },
          {
            "date": "2026-09-30",
            "close": 233.1,
            "volume": 55600000
          }
        ]
      },
      { ...


Same information, 871 characters versus something much longer, and now you can see the shape. `indent=2` is the first thing to reach for when a payload lands in your lap.

**TODO:** in one sentence, why is it wrong to say the API "sent you a dictionary"?

The API sent JSON text, not a Python dictionary. Therefore, it is better to use json.loads() to turn it into a dictionary.

### Step 2 — map the shape before you touch it

Never normalize a payload you have not walked. You need two numbers before you go any further: how many holdings there are, and how many price rows you should end up with.

The first line is done for you — `.keys()` gives you an object's names when you do not already know them. The counting is yours.

**TODO:** print the portfolio name, the number of holdings, and the total number of price records across all holdings.

In [3]:
print('top-level keys:', list(data.keys()))

print('portfolio name:', data['portfolio']['name'])
print('number of holdings:', len(data['portfolio']['holdings']))
print('total price records:', sum(len(h['prices']) for h in data['portfolio']['holdings']))

top-level keys: ['as_of', 'currency', 'source', 'portfolio']
portfolio name: Cavalier Growth
number of holdings: 3
total price records: 9


**TODO:** write your two numbers down here, because every assertion below checks against them.

- holdings: the number it prints
- price rows: the number it prints
- one row of the table you are about to build is one price record for one holding.

### Step 3 — trace a path, and hit the wall

A path is the sequence of names and positions that gets you to one value. Reading a path is the skill; the brackets are just typing.

**TODO:** print Apple's sector, then try to print Coca-Cola's the same way. One of those raises an exception. Catch it and read what it says. Then rewrite the line so it checks whether the name is there first and prints `'unknown'` when it is not — the `'pm25' not in r` test from Monday's missing/null/empty/zero table, pointed at `'sector'`.

In [4]:
print(data['portfolio']['holdings'][0]['company']['sector'])

try:
    print(data['portfolio']['holdings'][1]['company']['sector'])
except KeyError as e:
    print("Error:", e)

h = data['portfolio']['holdings'][1]
print(h['company']['sector'] if 'sector' in h['company'] else 'unknown')

Technology
Technology
Technology


### Step 4 — the four checks

It decoded and it parsed, so checks 1 and 2 passed the moment `json.loads` returned. Check 3 asks whether the data matches what you were promised, and check 4 asks whether it means what you think.

This payload has **four** planted problems. Three of them are in the price records and one is in a company object. None of them will raise an error.

**TODO:** walk every price record and report, with the ticker and date for each:

1. a `close` that came through as `None`
2. a `close` that is a *string* when every other one is a number
3. a `volume` of exactly `0`
4. a holding whose company object is missing a field the others have

Then answer the one that matters: **which of those four are genuinely missing data, and which are real measurements?** They do not get handled the same way.

In [5]:
for h in data['portfolio']['holdings']:
    for p in h['prices']:
        if p['close'] is None:
            print("None close:", h['ticker'], p['date'])

        if isinstance(p['close'], str):
            print("String close:", h['ticker'], p['date'])

        if p['volume'] == 0:
            print("Zero volume:", h['ticker'], p['date'])

    if 'sector' not in h['company']:
        print("Missing sector:", h['ticker'])

String close: MSFT 2026-09-29
Zero volume: MSFT 2026-09-29
None close: KO 2026-09-29
Missing sector: KO


**TODO:** your answer to the last question: The None close and missing sector are genuinely missing data, while the string close and zero volume are real measurements that should not automatically be treated as missing.

### Step 5 — flatten to one row per ticker per day

`record_path` is the list you want one row from. `meta` is everything above it that should be copied down onto each of those rows. `sector` only exists on two of the three companies, so you will need `errors='ignore'` to keep `json_normalize` from refusing.

*Expected: 9 rows.*

In [6]:
px = pd.json_normalize(
    data['portfolio']['holdings'],
    record_path='prices',
    meta=['ticker', ['company', 'name'], ['company', 'sector']],
    sep='.',
    errors='ignore'
)

assert len(px) == 9
print(px.dtypes)

date              object
close             object
volume             int64
ticker            object
company.name      object
company.sector    object
dtype: object


### Step 6 — fix the type drift

One string in a column of numbers is enough to make the whole column text, and a text column will not average. `pd.to_numeric(..., errors='coerce')` converts what it can and turns the rest into `NaN`.

**TODO:** convert `close` to a number, then confirm the dtype changed and count how many nulls you are left with. Say out loud where that null came from — it is not the string.

In [7]:
px['close'] = pd.to_numeric(px['close'], errors='coerce')

print(px['close'].dtype)
print(px['close'].isna().sum())

float64
1


### Step 7 — answer the questions

**TODO:** produce each of these, and report `count` next to every average so a reader can see how much data it rests on.

1. Mean closing price per ticker, and the number of days each mean is built from.
2. Each position's value on **2026-09-30** — that is `shares × close` — plus the total portfolio value and which holding is the largest.
3. Mean daily volume across all nine rows. Then compute it again with the zero-volume day excluded and report the difference.

That third one is the real question of the day. **Was dropping the zero the right call?** Defend your answer in a sentence.

In [8]:
# 1 — mean close and number of days per ticker
print(px.groupby('ticker')['close'].agg(['mean', 'count']))

# 2 — position values on 2026-09-30
latest = px[px['date'] == '2026-09-30'].copy()

shares = {h['ticker']: h['shares'] for h in data['portfolio']['holdings']}
latest['shares'] = latest['ticker'].map(shares)
latest['value'] = latest['shares'] * latest['close']

print(latest[['ticker', 'value']])
print('Total portfolio value:', latest['value'].sum())
print('Largest holding:', latest.loc[latest['value'].idxmax(), 'ticker'])

# 3 — mean volume with and without the zero
print('Mean volume:', px['volume'].mean())
print('Mean volume without zero:', px.loc[px['volume'] != 0, 'volume'].mean())
print('Difference:', px.loc[px['volume'] != 0, 'volume'].mean() - px['volume'].mean())

           mean  count
ticker                
AAPL    231.450      3
KO       71.075      2
MSFT    502.050      3
  ticker    value
2   AAPL  27972.0
5   MSFT  22738.5
8     KO  21270.0
Total portfolio value: 71980.5
Largest holding: AAPL
Mean volume: 26677777.777777776
Mean volume without zero: 30012500.0
Difference: 3334722.222222224


**TODO:** the zero-volume question: Dropping the zero-volume day was not appropriate because zero is a real measurement, so it should be included when calculating the overall mean volume.

### Step 8 — write the result back out

Your output is somebody else's input, so send the context with it — the date and the currency, not a bare table of numbers.

**TODO:** write your per-ticker summary to `portfolio_summary.json` with `indent=2` and `allow_nan=False`, then print the file back. If `allow_nan=False` raises, that is the point of it — a `NaN` reached your output and you have to decide what it should say before it leaves your machine.

In [9]:
summary = {
    'as_of': data['as_of'],
    'currency': data['currency'],
    'summary': px.groupby('ticker')['close'].agg(['mean', 'count']).reset_index().to_dict('records')
}

with open('portfolio_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, allow_nan=False)

with open('portfolio_summary.json', 'r') as f:
    print(f.read())

{
  "as_of": "2026-09-30",
  "currency": "USD",
  "summary": [
    {
      "ticker": "AAPL",
      "mean": 231.45000000000002,
      "count": 3
    },
    {
      "ticker": "KO",
      "mean": 71.075,
      "count": 2
    },
    {
      "ticker": "MSFT",
      "mean": 502.05,
      "count": 3
    }
  ]
}


---

## Part 2 — a second payload, if you have time

This one is already formatted, and it is deeper: a game-day response nested three levels down. Nothing here is new — it is the same `json_normalize` from Step 5 reaching further down, then the `merge` you have been using since Sep 14. The only twist is that you pull **two** tables out of one payload and join them back together.

In [10]:
response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

vendors: 4
line items: 6


Four vendors, six line items.

**TODO A.** Flatten to one row per vendor, carrying down the game, the date, and the zone's `zone` and `capacity`. Use `sep='_'` and leave the column names it gives you alone. *4 rows.*

**TODO B.** One vendor is missing a field. Find it, and add a boolean column recording whether revenue was reported. Do **not** fill it with zero — write a comment saying why that would be wrong here.

**TODO C.** Flatten the same payload again, down to line items this time, keeping `vendor_id` and `zone`. *6 rows, quantities summing to 755.*

**TODO D.** Join the two, so each line item carries its vendor name and zone capacity. The vendor ID is not spelled the same way in both tables, so reach for `left_on` and `right_on` the way you did on Sep 16. Use `indicator=True` and confirm you still have 6 rows with nothing unmatched.

In [12]:
# A — one row per vendor
vendors = pd.json_normalize(
    response['zones'],
    record_path='vendors',
    meta=['zone', 'capacity'],
    sep='_'
)

vendors['game'] = response['game']
vendors['date'] = response['date']

print(vendors)

# B — check whether revenue was reported
vendors['revenue_reported'] = vendors['sales_revenue'].notna()

# Do not replace missing revenue with 0 because missing means
# the revenue was not reported, not that revenue was $0.

# C — one row per line item
lines = pd.json_normalize(
    response['zones'],
    record_path=['vendors', 'lines'],
    meta=['zone', ['vendors', 'vendor_id']],
    sep='_'
)

print(lines)

# D — join the two tables
joined = pd.merge(
    lines,
    vendors,
    left_on='vendors_vendor_id',
    right_on='vendor_id',
    how='left',
    indicator=True
)

print(joined)
print(joined['_merge'].value_counts())

  vendor_id             name  \
0      V-01     Hoos Burgers   
1      V-07     Stadium Dogs   
2      V-12      Wahoo Wings   
3      V-18  Rally Rain Gear   

                                               lines  sales_orders  \
0  [{'item': 'Cheeseburger', 'qty': 180}, {'item'...           300   
1                  [{'item': 'Hot Dog', 'qty': 210}]           210   
2  [{'item': 'Wings 6pc', 'qty': 90}, {'item': 'W...           150   
3               [{'item': 'Rain Poncho', 'qty': 95}]            95   

   sales_revenue zone capacity         game        date  
0         2250.0    A      400  UVA vs Duke  2026-10-10  
1            NaN    A      400  UVA vs Duke  2026-10-10  
2         1800.0    B      150  UVA vs Duke  2026-10-10  
3          570.0    C      120  UVA vs Duke  2026-10-10  
           item  qty zone vendors_vendor_id
0  Cheeseburger  180    A              V-01
1       Hot Dog  120    A              V-01
2       Hot Dog  210    A              V-07
3     Wings 6pc   90  

---

## Checkpoint (participation)

Report your row counts from the portfolio payload and what you decided about the zero-volume day.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [17]:
# Checkpoint
holdings_count = 3
price_rows = 9
largest_position = 'AAPL'
null_vs_zero = 'None close is missing data; zero volume is a real measurement'
zero_volume_call = 'keep zero in the mean because zero is a real measurement'

print('holdings:', holdings_count)
print('price rows:', price_rows)
print('largest position:', largest_position)
print('null vs zero:', null_vs_zero)
print('zero volume:', zero_volume_call)

holdings: 3
price rows: 9
largest position: AAPL
null vs zero: None close is missing data; zero volume is a real measurement
zero volume: keep zero in the mean because zero is a real measurement
